# Notebook 3: Hailo Parsing & INT8 Quantization

This notebook converts the ONNX model into Hailo's format and quantizes it to INT8 using
Post-Training Quantization (PTQ).

**Pipeline:**
1. Parse ONNX → Hailo Archive (HAR) with `ClientRunner`
2. Apply ALLS model script (normalization + NMS configuration for 11 document classes)
3. Prepare calibration dataset (~200 diverse images)
4. Run INT8 quantization via `runner.optimize()`
5. Compare FP32 vs quantized inference quality

**Key concepts:**
- **ALLS script:** Hailo's model configuration language for normalization, NMS, and layer control
- **Calibration:** Uses a small set of representative images to compute activation statistics
  for mapping FP32 values to INT8. Diversity matters more than volume.
- **NMS replacement:** We strip PyTorch NMS and use Hailo's hardware-accelerated NMS instead

## 1. Setup & Imports

In [ ]:
import os
import time
import numpy as np
from pathlib import Path
from PIL import Image

import matplotlib.pyplot as plt
import matplotlib.patches as patches

from hailo_sdk_client import ClientRunner, InferenceContext

WORKSPACE = Path("/home/appuser/workspace")
MODELS_DIR = WORKSPACE / "models"
DATASET_DIR = WORKSPACE / "datasets" / "doclaynet"

print("Hailo SDK client imported successfully.")

## 2. Configuration

In [ ]:
# Hailo target hardware
HW_ARCH = "hailo8l"

# Model settings
NET_NAME = "doclaynet_yolo26n"
ONNX_PATH = str(MODELS_DIR / "best_simplified.onnx")
INPUT_SHAPE = (1, 3, 640, 640)  # NCHW

# Output paths
HAR_PATH = str(MODELS_DIR / f"{NET_NAME}.har")
QUANTIZED_HAR_PATH = str(MODELS_DIR / f"{NET_NAME}_quantized.har")

# Calibration settings
CALIB_SIZE = 200  # Number of calibration images
TARGET_SIZE = (640, 640)

# DocLayNet classes
NUM_CLASSES = 11
CLASS_NAMES = [
    "Caption", "Footnote", "Formula", "List-item", "Page-footer",
    "Page-header", "Picture", "Section-header", "Table", "Text", "Title"
]

# Verify ONNX model exists
assert os.path.exists(ONNX_PATH), f"ONNX model not found: {ONNX_PATH}. Run Notebook 2 first."
print(f"Hardware target: {HW_ARCH}")
print(f"ONNX model: {ONNX_PATH}")
print(f"Calibration images: {CALIB_SIZE}")

## 3. Prepare Calibration Dataset

Load a diverse set of images from the training set prepared in Notebook 1.
These are used purely for activation statistics — no labels are needed.

The calibration dataset must be a numpy array of shape `(N, H, W, 3)` in float32.

In [ ]:
def load_calibration_images(img_dir, target_size, max_images):
    """Load and preprocess images for Hailo calibration."""
    img_files = sorted([
        os.path.join(img_dir, f) for f in os.listdir(img_dir)
        if f.lower().endswith((".png", ".jpg", ".jpeg"))
    ])

    # Take evenly spaced samples for diversity
    if len(img_files) > max_images:
        step = len(img_files) / max_images
        img_files = [img_files[int(i * step)] for i in range(max_images)]
    else:
        img_files = img_files[:max_images]

    images = []
    for fpath in img_files:
        img = Image.open(fpath).convert("RGB")
        img = img.resize(target_size, Image.BILINEAR)
        images.append(np.array(img).astype(np.float32))

    calib_data = np.stack(images, axis=0)  # (N, H, W, 3)
    return calib_data


calib_img_dir = str(DATASET_DIR / "images" / "train")
calib_dataset = load_calibration_images(calib_img_dir, TARGET_SIZE, CALIB_SIZE)

print(f"Calibration dataset shape: {calib_dataset.shape}")
print(f"Value range: [{calib_dataset.min():.0f}, {calib_dataset.max():.0f}]")
print(f"Dtype: {calib_dataset.dtype}")
print(f"Memory: {calib_dataset.nbytes / (1024**2):.1f} MB")

In [ ]:
# Show a few calibration images
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
indices = np.linspace(0, len(calib_dataset) - 1, 5, dtype=int)
for ax, idx in zip(axes, indices):
    ax.imshow(calib_dataset[idx].astype(np.uint8))
    ax.set_title(f"Calib #{idx}", fontsize=9)
    ax.axis("off")
plt.suptitle("Calibration Dataset Samples", fontsize=14)
plt.tight_layout()
plt.show()

## 4. Parse ONNX to Hailo Format

The `ClientRunner` translates the ONNX graph into Hailo's internal HN + NPZ representation
and saves it as a HAR (Hailo Archive) file.

In [ ]:
# Initialize ClientRunner
runner = ClientRunner(hw_arch=HW_ARCH)

# Parse the ONNX model
# Note: If NMS nodes cause issues, use end_node_names to exclude them.
# The exact node names can be found from Notebook 2's graph inspection.
print("Parsing ONNX model...")
t0 = time.time()

hn, npz = runner.translate_onnx_model(
    ONNX_PATH,
    NET_NAME,
    # Uncomment and fill these if parsing fails due to NMS nodes:
    # end_node_names=["node_name_before_nms"],
    # net_input_shapes={"images": [1, 3, 640, 640]},
)

parse_time = time.time() - t0
print(f"Parsing completed in {parse_time:.1f}s")

# Save initial HAR
runner.save_har(HAR_PATH)
har_size_mb = os.path.getsize(HAR_PATH) / (1024 * 1024)
print(f"Initial HAR saved: {HAR_PATH} ({har_size_mb:.1f} MB)")

## 5. Apply ALLS Model Script

The ALLS (Accelerated Layer Library Script) configures:
- **Normalization:** Divides pixel values by 255 so inference input can be uint8 [0-255]
- **NMS postprocess:** Replaces PyTorch NMS with Hailo's hardware-accelerated version

YOLO26 uses the same detection head architecture as YOLOv8, so `meta_arch=yolov8` is correct.

In [ ]:
# Define the ALLS model script
alls_script = f"""
normalization1 = normalization([0.0, 0.0, 0.0], [255.0, 255.0, 255.0])
nms_postprocess(meta_arch=yolov8, engine=hailo, classes={NUM_CLASSES}, confidence_threshold=0.25, iou_threshold=0.45)
"""

print("ALLS Model Script:")
print(alls_script)

# Load the script into the runner
runner.load_model_script(alls_script)
print("Model script loaded successfully.")

## 6. INT8 Quantization (PTQ)

This is the core step: the optimizer analyzes the calibration data to determine optimal
quantization parameters (scale and zero-point) for each layer, converting FP32 weights
and activations to INT8.

This step may take several minutes depending on model size and calibration set.

In [ ]:
print(f"Starting INT8 quantization with {len(calib_dataset)} calibration images...")
print("This may take several minutes.\n")

t0 = time.time()
runner.optimize(calib_dataset)
quant_time = time.time() - t0

print(f"\nQuantization completed in {quant_time:.1f}s ({quant_time/60:.1f} min)")

In [ ]:
# Save quantized HAR
runner.save_har(QUANTIZED_HAR_PATH)
quant_har_size_mb = os.path.getsize(QUANTIZED_HAR_PATH) / (1024 * 1024)

print(f"Quantized HAR saved: {QUANTIZED_HAR_PATH}")
print(f"Quantized HAR size: {quant_har_size_mb:.1f} MB")
print(f"\nSize comparison:")
print(f"  Initial HAR:    {har_size_mb:.1f} MB")
print(f"  Quantized HAR:  {quant_har_size_mb:.1f} MB")

## 7. Quantization Quality Check

Compare inference outputs between FP32 and INT8 quantized contexts to assess
accuracy degradation from quantization.

In [ ]:
# Load a few test images for comparison
val_img_dir = DATASET_DIR / "images" / "validation"
test_files = sorted(os.listdir(val_img_dir))[:5]

test_images = []
for fname in test_files:
    img = Image.open(val_img_dir / fname).convert("RGB").resize(TARGET_SIZE)
    test_images.append(np.array(img).astype(np.float32))

test_batch = np.stack(test_images)  # (N, 640, 640, 3)
print(f"Test batch shape: {test_batch.shape}")

In [ ]:
# Run inference in different contexts
print("Running FP32 inference...")
with runner.infer_context(InferenceContext.SDK_FP_OPTIMIZED) as ctx:
    fp32_outputs = runner.infer(ctx, test_batch)

print("Running quantized (INT8) inference...")
with runner.infer_context(InferenceContext.SDK_QUANTIZED) as ctx:
    int8_outputs = runner.infer(ctx, test_batch)

# Compare outputs
print(f"\nFP32 output keys: {list(fp32_outputs.keys()) if isinstance(fp32_outputs, dict) else type(fp32_outputs)}")
print(f"INT8 output keys: {list(int8_outputs.keys()) if isinstance(int8_outputs, dict) else type(int8_outputs)}")

# Compute numerical difference
if isinstance(fp32_outputs, dict) and isinstance(int8_outputs, dict):
    for key in fp32_outputs:
        if key in int8_outputs:
            fp_out = fp32_outputs[key]
            q_out = int8_outputs[key]
            mae = np.mean(np.abs(fp_out - q_out))
            rmse = np.sqrt(np.mean((fp_out - q_out) ** 2))
            cosine = np.dot(fp_out.flatten(), q_out.flatten()) / (
                np.linalg.norm(fp_out) * np.linalg.norm(q_out) + 1e-8
            )
            print(f"\n  Output '{key}':")
            print(f"    MAE:             {mae:.6f}")
            print(f"    RMSE:            {rmse:.6f}")
            print(f"    Cosine Similarity: {cosine:.6f}")
else:
    fp_arr = np.array(fp32_outputs).flatten()
    q_arr = np.array(int8_outputs).flatten()
    mae = np.mean(np.abs(fp_arr - q_arr))
    cosine = np.dot(fp_arr, q_arr) / (np.linalg.norm(fp_arr) * np.linalg.norm(q_arr) + 1e-8)
    print(f"\n  MAE: {mae:.6f}")
    print(f"  Cosine Similarity: {cosine:.6f}")

In [ ]:
print("=" * 60)
print("QUANTIZATION SUMMARY")
print("=" * 60)
print(f"Hardware target:     {HW_ARCH}")
print(f"Model name:          {NET_NAME}")
print(f"Calibration images:  {CALIB_SIZE}")
print(f"Parse time:          {parse_time:.1f}s")
print(f"Quantization time:   {quant_time:.1f}s")
print(f"Initial HAR:         {har_size_mb:.1f} MB")
print(f"Quantized HAR:       {quant_har_size_mb:.1f} MB")
print(f"\nQuantized model saved to: {QUANTIZED_HAR_PATH}")
print(f"\nNext step: Run 04_Compilation_and_Profiling.ipynb")